<a href="https://colab.research.google.com/github/YuYeon-K/AI_MentalHealth_Chatbot/blob/main/NLP_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Run this code to get started! This'll load some packages and set up some dependencies for us
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
!pip install Keras-Preprocessing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import spacy
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers , activations , models , preprocessing, utils
from keras import Input, Model
from keras.activations import softmax
from keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import RMSprop
from keras_preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from keras_preprocessing.text import Tokenizer
tf.random.set_seed(1)
from google.colab import drive
import time
import os
import gdown

# set pandas viewing options
pd.set_option("display.max_columns", None)
pd.set_option('display.width', None)
pd.set_option("max_colwidth", None)
#pd.reset_option("max_colwidth")

# the source of our data is: https://github.com/nbertagnolli/counsel-chat

# load our weights
!wget -q --show-progress "https://storage.googleapis.com/inspirit-ai-data-bucket-1/Data/AI%20Scholars/Sessions%206%20-%2010%20(Projects)/Project%20-%20Mental%20Health%20Chatbots/chatbot_seq2seq_v3.h5"


In [1]:
#@title Run this cell to read in our dataset
chat_data = pd.read_csv("https://raw.githubusercontent.com/nbertagnolli/counsel-chat/master/data/20200325_counsel_chat.csv")

# !pip install better_profanity

# from better_profanity import profanity

# def filter_profanity(df: pd.DataFrame) -> pd.DataFrame:
#     profanity.load_censor_words()

#     # The applymap function applies a function to every cell of the dataframe
#     # The lambda function returns True if the cell contains profanity, False otherwise
#     contains_profanity = df[['answerText', 'questionText']].applymap(lambda x: profanity.contains_profanity(x))

#     # Filter out the rows that contain profanity
#     df = df[~(contains_profanity['answerText'] | contains_profanity['questionText'])]

#     return df

# chat_data = filter_profanity(chat_data)

NameError: name 'pd' is not defined

In [ ]:
chat_data.head()

In [ ]:
#@title Run this cell to plot the number of questions and answers for each topic

chat_data["topic"].value_counts().plot.bar()
plt.title("Number of questions and answers for each topic")
plt.show()


In [ ]:
X = chat_data["questionText"]
y = chat_data["answerText"]


In [ ]:
def preprocess_text(phrase):
  phrase = re.sub(r"\xa0", "", phrase) # removes "\xa0"
  phrase = re.sub(r"\n", "", phrase) # removes "\n"
  phrase = re.sub("[.]{1,}", ".", phrase) # removes duplicate "."s
  phrase = re.sub("[ ]{1,}", " ", phrase) # removes duplicate spaces

  return phrase

In [ ]:
X = X.apply(preprocess_text)
y = y.apply(preprocess_text)

In [ ]:
#@title Run this cell to plot the lengths of questions and answers, in sentences
question_lengths, answer_lengths = [], []

for (question, answer) in zip(X, y):
  # split by "."
  question_arr = question.split(".")
  answer_arr = answer.split(".")

  # get length
  length_question = len(question_arr)
  length_answer = len(answer_arr)

  # add to array
  question_lengths.append(length_question)
  answer_lengths.append(length_answer)

# Display the plot
plt.hist(question_lengths, color = "red", alpha = 0.5)
plt.hist(answer_lengths, color = "blue", alpha = 0.2)
plt.title("Lengths of questions (in red) and answers (in blue)")
plt.axvline(np.mean(question_lengths), color = "red")
plt.axvline(np.mean(answer_lengths), color = "blue")
plt.show()

In [ ]:
question_answer_pairs = []

MAX_LENGTH = 100 # the maximum length for our sequences

for (question, answer) in zip(X, y):
  question = preprocess_text(question)
  answer = preprocess_text(answer)

  # split up question and answer into their constituent sentences

  question_arr = question.split(".")
  answer_arr = answer.split(".")

  # get the maximum number of question/answer pairs we can form,
  # which will be the shorter of len(question_arr) and len(answer_arr)

  max_sentences = min(len(question_arr), len(answer_arr))

  for i in range(max_sentences):
    q_a_pair = []

    # get maximum sentence length
    max_q_length = min(MAX_LENGTH, len(question_arr[i]))
    max_a_length = min(MAX_LENGTH, len(answer_arr[i]))

    # append question, answer to pair (e.g,. first sentence of question + first sentence of answer, etc.)
    question_to_append = question_arr[i][0:max_q_length]
    q_a_pair.append(question_to_append)

    answer_to_append = "<START> " + answer_arr[i][0:max_a_length] + " <END>"
    q_a_pair.append(answer_to_append)

    question_answer_pairs.append(q_a_pair)

In [ ]:

def tokenize(sentence):
  tokens = sentence.split(" ")
  return tokens

In [ ]:
#@title Enter a sentence to test your tokenization function

sentence = "I love Inspirit AI!" #@param {type:"string"}

# split sentence by space
split_sentence = tokenize(sentence)
print(f"The original sentence: {sentence}")
print(f"The tokenized sentence: {split_sentence}")

In [ ]:
def tokenize_and_pad(sentence, max_len):
  """
    Tokenizes our sentence (splits up the individual words), adds <SOS> and <EOS> tags,
    and, if it's too short, add, padding at the end (before <EOS>)
  """

  sentence_arr = sentence.split(" ")

  diff = max_len - (len(sentence_arr) + 2)

  if diff > 0: # if too short, add padding + start/end tokens
    tokenized_sentence = ["<SOS>"] + sentence_arr + ["<EOS>"] + (["<pad>"] * diff)
  elif diff == 0: # if right length, just add start/end tokens
    tokenized_sentence = ["<SOS>"] + sentence_arr + ["<EOS>"]
  else: # if too long, add start/end tokens, truncate
    tokenized_sentence = ["<SOS>"] + sentence_arr[0:diff] + ["<EOS>"]

  return tokenized_sentence


In [ ]:
#@title Run this code chunk to see your function at work! Enter a sentence and a maximum length and see how it looks when tokenized and given padding

sentence = "Hello from the other side!" #@param {type:"string"}
max_len =  10#@param {type:"integer"}

print(tokenize_and_pad(sentence, max_len))

In [ ]:
#@title Run this cell to do some initialization

# re-create questions, answers
questions = [arr[0] for arr in question_answer_pairs]
answers = [arr[1] for arr in question_answer_pairs]

target_regex = '!"#$%&()*+,-./:;<=>?@[\]^_`{|}~\t\n\'0123456789'
tokenizer = Tokenizer(filters=target_regex)
tokenizer.fit_on_texts(questions + answers)
VOCAB_SIZE = len(tokenizer.word_index) + 1

# create encoder input data
tokenized_questions = tokenizer.texts_to_sequences(questions)
maxlen_questions = max([len(x) for x in tokenized_questions])
#encoder_input_data = pad_sequences(tokenized_questions,
#                                   maxlen=maxlen_questions,
#                                   padding='post')

# create decoder input data
tokenized_answers = tokenizer.texts_to_sequences(answers)
maxlen_answers = max([len(x) for x in tokenized_answers])
#decoder_input_data = pad_sequences(tokenized_answers,
#                                   maxlen=maxlen_answers,
#                                   padding='post')

# create decoder output data
#for i in range(len(tokenized_answers)):
#    tokenized_answers[i] = tokenized_answers[i][1:]
#padded_answers = pad_sequences(tokenized_answers, maxlen=maxlen_answers, padding='post')
#decoder_output_data = to_categorical(padded_answers, VOCAB_SIZE)

In [ ]:
enc_inputs = Input(shape=(None,))
enc_embedding = Embedding(VOCAB_SIZE, 200, mask_zero=True)(enc_inputs)
enc_lstm = LSTM(200, return_state=True)
_, state_h, state_c = enc_lstm(enc_embedding)
enc_states = [state_h, state_c]

dec_inputs = Input(shape=(None,))
dec_embedding = Embedding(VOCAB_SIZE, 200, mask_zero=True)(dec_inputs)
dec_lstm = LSTM(200, return_state=True, return_sequences=True)
dec_outputs, _, _ = dec_lstm(dec_embedding, initial_state=enc_states)

dec_dense = Dense(VOCAB_SIZE, activation=softmax)
output = dec_dense(dec_outputs)

In [ ]:

model = Model([enc_inputs, dec_inputs], output)

model.compile(optimizer=RMSprop(), loss='categorical_crossentropy')

model.summary()

tf.keras.backend.clear_session()

In [ ]:
path_to_weight = "chatbot_seq2seq_v3.h5"

model.load_weights(path_to_weight)

In [ ]:
#@title Run this cell to do some initialization
def make_inference_models():
    dec_state_input_h = Input(shape=(200,), name="input_h")
    dec_state_input_c = Input(shape=(200,), name="input_c")
    dec_states_inputs = [dec_state_input_h, dec_state_input_c]
    dec_outputs, state_h, state_c = dec_lstm(dec_embedding,
                                             initial_state=dec_states_inputs)
    dec_states = [state_h, state_c]
    dec_outputs = dec_dense(dec_outputs)
    dec_model = Model(
        inputs=[dec_inputs] + dec_states_inputs,
        outputs=[dec_outputs] + dec_states)
    print('Inference decoder:')
    dec_model.summary()
    print('Inference encoder:')
    enc_model = Model(inputs=enc_inputs, outputs=enc_states)
    enc_model.summary()
    return enc_model, dec_model

def str_to_tokens(sentence: str):
    words = sentence.lower().split()
    tokens_list = list()
    for current_word in words:
        result = tokenizer.word_index.get(current_word, '')
        if result != '':
            tokens_list.append(result)
    return pad_sequences([tokens_list],
                         maxlen=maxlen_questions,
                         padding='post')

enc_model, dec_model = make_inference_models()


In [ ]:
#@title Run this cell and type your question to see what our chatbot says!

question = "Could you give me some advice?" #@param {type:"string"}
for _ in range(1):
    states_values = enc_model.predict(
        str_to_tokens(question))
    empty_target_seq = np.zeros((1, 1))
    empty_target_seq[0, 0] = tokenizer.word_index['start']
    stop_condition = False
    decoded_translation = ''
    while not stop_condition:
        dec_outputs, h, c = dec_model.predict([empty_target_seq]
                                              + states_values)
        sampled_word_index = np.argmax(dec_outputs[0, -1, :])
        sampled_word = None
        for word, index in tokenizer.word_index.items():
            if sampled_word_index == index:
                if word != 'end':
                    decoded_translation += ' {}'.format(word)
                sampled_word = word

        if sampled_word == 'end' \
                or len(decoded_translation.split()) \
                > maxlen_answers:
            stop_condition = True

        empty_target_seq = np.zeros((1, 1))
        empty_target_seq[0, 0] = sampled_word_index
        states_values = [h, c]

    print(decoded_translation)